#### 25 PERCENT OVERSAMPLING

In [1]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import ADASYN
from sklearn.model_selection import train_test_split
from collections import Counter
import os

# Load original dataset
df = pd.read_csv("full_data_unhealthy_imputed_reduced_enhanced.csv")

# Define target and feature columns
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour_bin']

# Feature engineering
def create_features(X):
    X = X.copy()
    X['hour_sin'] = np.sin(2 * np.pi * X['hour_bin'] / 24)
    X['hour_cos'] = np.cos(2 * np.pi * X['hour_bin'] / 24)
    X['eat_rest_ratio'] = X['EAT'] / (X['REST'] + 1e-6)
    X['activity_rest_ratio'] = np.abs(X['ACTIVITY_LEVEL']) / (X['REST'] + 1e-6)
    return X.drop(columns=['hour_bin'])

X = create_features(df[features])
y = df[targets]

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ADASYN resampling function
def adasyn_resample_custom(X, y, target_ratio=0.25):
    X_res_list = []
    y_res_list = []

    for target in targets:
        print(f"\nResampling target: {target}")
        counts = Counter(y[target])
        majority_class = counts[0]
        desired_minority = int(majority_class * target_ratio / (1 - target_ratio))
        print(f"Original counts: {counts}, Desired minority: {desired_minority}")

        sampler = ADASYN(sampling_strategy={1: desired_minority}, random_state=42, n_neighbors=5)
        X_res, y_res = sampler.fit_resample(X, y[target])

        X_res_df = pd.DataFrame(X_res, columns=X.columns)
        y_res_series = pd.Series(y_res, name=target)

        X_res_list.append(X_res_df)
        y_res_list.append(y_res_series)

    return X_res_list, y_res_list

# Run resampling
X_resampled_list, y_resampled_list = adasyn_resample_custom(X_train, y_train, target_ratio=0.25)

# Save datasets
output_dir = "adasyn_25pct"
os.makedirs(output_dir, exist_ok=True)

for i, target in enumerate(targets):
    df_combined = pd.concat([X_resampled_list[i], y_resampled_list[i]], axis=1)
    df_combined.to_csv(f"{output_dir}/adasyn_25pct_{target}.csv", index=False)
    print(f"Saved: {output_dir}/adasyn_25pct_{target}.csv | Shape: {df_combined.shape}")



Resampling target: oestrus
Original counts: Counter({0: 253490, 1: 1042}), Desired minority: 84496

Resampling target: calving
Original counts: Counter({0: 253961, 1: 571}), Desired minority: 84653

Resampling target: lameness
Original counts: Counter({0: 254115, 1: 417}), Desired minority: 84705

Resampling target: mastitis
Original counts: Counter({0: 254406, 1: 126}), Desired minority: 84802
Saved: adasyn_25pct/adasyn_25pct_oestrus.csv | Shape: (337926, 9)
Saved: adasyn_25pct/adasyn_25pct_calving.csv | Shape: (338821, 9)
Saved: adasyn_25pct/adasyn_25pct_lameness.csv | Shape: (338896, 9)
Saved: adasyn_25pct/adasyn_25pct_mastitis.csv | Shape: (339168, 9)


#### RANDOM FOREST AND LIGHT GBM APPLIED

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import classification_report, make_scorer, f1_score
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
import joblib
import os

# --------------------------
# Configuration
# --------------------------
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
data_dir = "adasyn_25pct"
model_dir = "models/25pct"
os.makedirs(model_dir, exist_ok=True)

# --------------------------
# Custom scorer (weighted F1)
# --------------------------
scorer = make_scorer(lambda yt, yp: f1_score(yt, yp, average="weighted"))
cv = KFold(n_splits=3, shuffle=True, random_state=42)

# --------------------------
# Training function
# --------------------------
def tune_and_train(X_train, y_train, model_type):
    if model_type == "rf":
        model = RandomForestClassifier(random_state=42, n_jobs=-1)
        param_grid = {
            "n_estimators": [100, 150],
            "max_depth": [10, 20],
            "min_samples_split": [2, 5],
            "class_weight": ["balanced"]
        }
    else:  # LightGBM
        model = LGBMClassifier(random_state=42, n_jobs=-1)
        param_grid = {
            "n_estimators": [200, 300],
            "max_depth": [12, 16],
            "learning_rate": [0.05, 0.1],
            "subsample": [0.8],
            "colsample_bytree": [0.8],
            "class_weight": ["balanced"]
        }

    grid = GridSearchCV(
        model,
        param_grid,
        scoring=scorer,
        cv=cv,
        verbose=2,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    print(f"Best parameters for {model_type.upper()}: {grid.best_params_}")
    return grid.best_estimator_

# --------------------------
# Loop over models & targets
# --------------------------
for model_type in ["rf", "lgbm"]:
    print(f"\nTraining {model_type.upper()} models (25% ADASYN):")

    for target in targets:
        print(f"\nTarget: {target}")
        df = pd.read_csv(f"{data_dir}/adasyn_25pct_{target}.csv")
        X = df.drop(columns=[target])
        y = df[target]

        # Split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42
        )

        # Train
        model = tune_and_train(X_train, y_train, model_type)

        # Evaluation
        y_pred = model.predict(X_test)
        print(f"\nClassification Report for {model_type.upper()} - {target}:\n")
        print(classification_report(y_test, y_pred, digits=4))

        # Save model
        path = os.path.join(model_dir, f"{model_type}_{target}.pkl")
        joblib.dump(model, path)
        print(f"Model saved to: {path}")



Training RF models (25% ADASYN):

Target: oestrus
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best parameters for RF: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 150}

Classification Report for RF - oestrus:

              precision    recall  f1-score   support

           0     0.9666    0.9468    0.9566     50699
           1     0.8495    0.9017    0.8748     16887

    accuracy                         0.9355     67586
   macro avg     0.9080    0.9242    0.9157     67586
weighted avg     0.9373    0.9355    0.9361     67586

Model saved to: models/25pct\rf_oestrus.pkl

Target: calving
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best parameters for RF: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 150}

Classification Report for RF - calving:

              precision    recall  f1-score   support

           0     0.9798    0.9673    0.9735     50793
           1     0

#### MLP APPLIED

In [5]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import joblib

# -------------------
# Configuration
# -------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGETS = ['oestrus', 'calving', 'lameness', 'mastitis']
DATA_DIR = "adasyn_25pct"
SAVE_DIR = "models/25pct_mlp"
os.makedirs(SAVE_DIR, exist_ok=True)
BATCH_SIZE = 256
EPOCHS = 30
LEARNING_RATE = 0.0005

# -------------------
# Dataset Class
# -------------------
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# -------------------
# MLP Model
# -------------------
class MLPModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.model(x)

# -------------------
# Training Function
# -------------------
def train_mlp_for_target(target):
    print(f"\nTraining MLP for: {target}")
    df = pd.read_csv(f"{DATA_DIR}/adasyn_25pct_{target}.csv")
    X = df.drop(columns=[target])
    y = df[target]

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Save scaler
    joblib.dump(scaler, os.path.join(SAVE_DIR, f"scaler_{target}.pkl"))

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=42
    )

    # DataLoaders
    train_loader = DataLoader(TabularDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(TabularDataset(X_test, y_test), batch_size=BATCH_SIZE)

    # Model
    model = MLPModel(input_dim=X.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    best_loss = float("inf")
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"mlp_{target}_25pct.pth"))

    # Evaluation
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(DEVICE)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            y_true.extend(y_batch.numpy())
            y_pred.extend(preds.cpu().numpy())

    print(f"\nClassification Report for {target}:\n")
    print(classification_report(y_true, y_pred, digits=4))

# -------------------
# Run for all targets
# -------------------
for target in TARGETS:
    train_mlp_for_target(target)



Training MLP for: oestrus
Epoch 1: Loss = 0.4558
Epoch 2: Loss = 0.4048
Epoch 3: Loss = 0.3840
Epoch 4: Loss = 0.3731
Epoch 5: Loss = 0.3684
Epoch 6: Loss = 0.3657
Epoch 7: Loss = 0.3621
Epoch 8: Loss = 0.3586
Epoch 9: Loss = 0.3549
Epoch 10: Loss = 0.3525
Epoch 11: Loss = 0.3517
Epoch 12: Loss = 0.3499
Epoch 13: Loss = 0.3492
Epoch 14: Loss = 0.3491
Epoch 15: Loss = 0.3466
Epoch 16: Loss = 0.3464
Epoch 17: Loss = 0.3459
Epoch 18: Loss = 0.3462
Epoch 19: Loss = 0.3453
Epoch 20: Loss = 0.3431
Epoch 21: Loss = 0.3432
Epoch 22: Loss = 0.3454
Epoch 23: Loss = 0.3449
Epoch 24: Loss = 0.3435
Epoch 25: Loss = 0.3445
Epoch 26: Loss = 0.3454
Epoch 27: Loss = 0.3452
Epoch 28: Loss = 0.3436
Epoch 29: Loss = 0.3449
Epoch 30: Loss = 0.3413

Classification Report for oestrus:

              precision    recall  f1-score   support

           0     0.8610    0.9826    0.9178     50699
           1     0.9092    0.5239    0.6647     16887

    accuracy                         0.8680     67586
   macr

#### TABNET APPLIED

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
import torch
import os

# ----------------------
# Configuration
# ----------------------
targets = ['oestrus', 'calving', 'lameness', 'mastitis']
data_dir = "adasyn_25pct"
model_dir = "models/25pct_tabnet"
os.makedirs(model_dir, exist_ok=True)

# ----------------------
# Training Loop
# ----------------------
for target in targets:
    print(f"\nTraining TabNet for target: {target} (25% ADASYN)")

    # Load data
    df = pd.read_csv(f"{data_dir}/adasyn_25pct_{target}.csv")
    X = df.drop(columns=[target])
    y = df[target]

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Encode target
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_test_enc = le.transform(y_test)

    # Define TabNet classifier
    clf = TabNetClassifier(
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=1e-2),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params={"step_size":10, "gamma":0.9},
        verbose=0,
        seed=42
    )

    # Fit model
    clf.fit(
        X_train=X_train_scaled, y_train=y_train_enc,
        eval_set=[(X_test_scaled, y_test_enc)],
        eval_name=["val"],
        eval_metric=["accuracy"],
        max_epochs=100,
        patience=10,
        batch_size=1024,
        virtual_batch_size=128
    )

    # Evaluate
    preds = clf.predict(X_test_scaled)
    print(f"\nClassification Report for {target}:\n")
    print(classification_report(y_test_enc, preds, digits=4))

    # Save model
    save_path = f"{model_dir}/tabnet_{target}_25pct"
    clf.save_model(save_path)
    print(f"Model saved to: {save_path}.zip")



Training TabNet for target: oestrus (25% ADASYN)

Early stopping occurred at epoch 12 with best_epoch = 2 and best_val_accuracy = 0.87314


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Classification Report for oestrus:

              precision    recall  f1-score   support

           0     0.8685    0.9791    0.9205     50699
           1     0.8983    0.5551    0.6862     16887

    accuracy                         0.8731     67586
   macro avg     0.8834    0.7671    0.8033     67586
weighted avg     0.8760    0.8731    0.8620     67586

Successfully saved model at models/25pct_tabnet/tabnet_oestrus_25pct.zip
Model saved to: models/25pct_tabnet/tabnet_oestrus_25pct.zip

Training TabNet for target: calving (25% ADASYN)

Early stopping occurred at epoch 26 with best_epoch = 16 and best_val_accuracy = 0.92419


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Classification Report for calving:

              precision    recall  f1-score   support

           0     0.9216    0.9824    0.9510     50793
           1     0.9344    0.7499    0.8321     16972

    accuracy                         0.9242     67765
   macro avg     0.9280    0.8662    0.8916     67765
weighted avg     0.9248    0.9242    0.9213     67765

Successfully saved model at models/25pct_tabnet/tabnet_calving_25pct.zip
Model saved to: models/25pct_tabnet/tabnet_calving_25pct.zip

Training TabNet for target: lameness (25% ADASYN)

Early stopping occurred at epoch 12 with best_epoch = 2 and best_val_accuracy = 0.86282


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Classification Report for lameness:

              precision    recall  f1-score   support

           0     0.8558    0.9827    0.9148     50824
           1     0.9066    0.5035    0.6474     16956

    accuracy                         0.8628     67780
   macro avg     0.8812    0.7431    0.7811     67780
weighted avg     0.8685    0.8628    0.8479     67780

Successfully saved model at models/25pct_tabnet/tabnet_lameness_25pct.zip
Model saved to: models/25pct_tabnet/tabnet_lameness_25pct.zip

Training TabNet for target: mastitis (25% ADASYN)

Early stopping occurred at epoch 37 with best_epoch = 27 and best_val_accuracy = 0.88582


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Classification Report for mastitis:

              precision    recall  f1-score   support

           0     0.8848    0.9747    0.9276     50882
           1     0.8908    0.6190    0.7304     16952

    accuracy                         0.8858     67834
   macro avg     0.8878    0.7969    0.8290     67834
weighted avg     0.8863    0.8858    0.8783     67834

Successfully saved model at models/25pct_tabnet/tabnet_mastitis_25pct.zip
Model saved to: models/25pct_tabnet/tabnet_mastitis_25pct.zip


#### LSTM APPLIED

In [14]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# -------------------
# Configuration
# -------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGETS = ['oestrus', 'calving', 'lameness', 'mastitis']
DATA_DIR = "adasyn_25pct"
MODEL_DIR = "models/25pct_lstm"
os.makedirs(MODEL_DIR, exist_ok=True)

EPOCHS = 10
BATCH_SIZE = 64
LR = 0.001

# -------------------
# LSTM Model
# -------------------
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # last time step
        out = self.fc(out)
        return self.sigmoid(out)

# -------------------
# Training Function
# -------------------
def train_lstm_model(X, y, target_name):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_scaled = np.expand_dims(X_scaled, axis=1)  # reshape for LSTM: (batch, seq_len=1, features)

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

    train_data = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                               torch.tensor(y_train.values, dtype=torch.float32))
    test_data = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                              torch.tensor(y_test.values, dtype=torch.float32))

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

    model = LSTMClassifier(input_dim=X_train.shape[2], hidden_dim=64, output_dim=1).to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            preds = model(xb).squeeze()
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        preds_all, labels_all = [], []
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(DEVICE)
                out = model(xb).squeeze()
                preds_all.extend((out > 0.5).int().cpu().numpy())
                labels_all.extend(yb.int().cpu().numpy())

        print(f"\nEpoch {epoch} - {target_name}:\n")
        print(classification_report(labels_all, preds_all))

    # Save model
    model_path = os.path.join(MODEL_DIR, f"lstm_{target_name}_25pct.pth")
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to: {model_path}\n")

# -------------------
# Train all targets
# -------------------
for target in TARGETS:
    df = pd.read_csv(f"{DATA_DIR}/adasyn_25pct_{target}.csv")
    X = df.drop(columns=[target])
    y = df[target]
    train_lstm_model(X, y, target)



Epoch 1 - oestrus:

              precision    recall  f1-score   support

           0       0.86      0.98      0.91     50699
           1       0.88      0.51      0.65     16887

    accuracy                           0.86     67586
   macro avg       0.87      0.74      0.78     67586
weighted avg       0.86      0.86      0.85     67586


Epoch 2 - oestrus:

              precision    recall  f1-score   support

           0       0.86      0.97      0.92     50699
           1       0.87      0.54      0.67     16887

    accuracy                           0.86     67586
   macro avg       0.87      0.76      0.79     67586
weighted avg       0.87      0.86      0.85     67586


Epoch 3 - oestrus:

              precision    recall  f1-score   support

           0       0.87      0.98      0.92     50699
           1       0.89      0.55      0.68     16887

    accuracy                           0.87     67586
   macro avg       0.88      0.76      0.80     67586
weighted av

#### TCN MODEL APPLIED

In [17]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.optim import Adam
from torch.nn.utils import weight_norm
import random

# Set random seed
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Config
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "adasyn_25pct"
MODEL_DIR = "models/25pct_tcn"
os.makedirs(MODEL_DIR, exist_ok=True)
TARGETS = ['oestrus', 'calving', 'lameness', 'mastitis']

# Dataset class
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# TCN block
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout):
        super().__init__()
        self.conv1 = weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCN(nn.Module):
    def __init__(self, input_size, output_size, num_channels, kernel_size=3, dropout=0.3):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = input_size if i == 0 else num_channels[i - 1]
            out_channels = num_channels[i]
            layers.append(TemporalBlock(in_channels, out_channels, kernel_size, stride=1,
                                        dilation=dilation_size, padding=(kernel_size - 1) * dilation_size,
                                        dropout=dropout))
        self.network = nn.Sequential(*layers)
        self.linear = nn.Linear(num_channels[-1], output_size)

    def forward(self, x):
        x = self.network(x)
        x = x[:, :, -1]
        return self.linear(x)

# Training function
def train_tcn_model(target):
    df = pd.read_csv(f"{DATA_DIR}/adasyn_25pct_{target}.csv")
    features = df.drop(columns=[target])
    labels = df[target]

    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # Reshape to [batch, channels, time]
    X = features_scaled.reshape(features_scaled.shape[0], features_scaled.shape[1], 1)
    X = np.transpose(X, (0, 2, 1))  # (batch, channels, time)

    X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, stratify=labels, random_state=42)

    train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=256, shuffle=True)
    test_loader = DataLoader(TimeSeriesDataset(X_test, y_test), batch_size=256)

    model = TCN(input_size=1, output_size=2, num_channels=[16, 32, 64], kernel_size=3, dropout=0.3).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=0.001)

    best_accuracy = 0
    for epoch in range(1, 11):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            output = model(xb)
            loss = criterion(output, yb)
            loss.backward()
            optimizer.step()

        # Evaluate
        model.eval()
        y_true, y_pred = [], []
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(DEVICE)
                outputs = model(xb)
                _, predicted = torch.max(outputs, 1)
                y_true.extend(yb.numpy())
                y_pred.extend(predicted.cpu().numpy())

        acc = (np.array(y_pred) == np.array(y_true)).mean()
        print(f"\nEpoch {epoch} - {target} Accuracy: {acc:.4f}")
        print(classification_report(y_true, y_pred, digits=4))

        if acc > best_accuracy:
            best_accuracy = acc
            model_path = os.path.join(MODEL_DIR, f"tcn_{target}_25pct.pth")
            torch.save(model.state_dict(), model_path)
            print(f"Model saved to: {model_path}")

# Train all targets
for target in TARGETS:
    train_tcn_model(target)

+

C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



Epoch 1 - oestrus Accuracy: 0.8563
              precision    recall  f1-score   support

           0     0.8571    0.9702    0.9101     50699
           1     0.8518    0.5144    0.6414     16887

    accuracy                         0.8563     67586
   macro avg     0.8545    0.7423    0.7758     67586
weighted avg     0.8558    0.8563    0.8430     67586

Model saved to: models/25pct_tcn\tcn_oestrus_25pct.pth

Epoch 2 - oestrus Accuracy: 0.8591
              precision    recall  f1-score   support

           0     0.8723    0.9515    0.9101     50699
           1     0.7997    0.5816    0.6735     16887

    accuracy                         0.8591     67586
   macro avg     0.8360    0.7666    0.7918     67586
weighted avg     0.8541    0.8591    0.8510     67586

Model saved to: models/25pct_tcn\tcn_oestrus_25pct.pth

Epoch 3 - oestrus Accuracy: 0.8666
              precision    recall  f1-score   support

           0     0.8675    0.9704    0.9160     50699
           1     0.

C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



Epoch 1 - calving Accuracy: 0.9098
              precision    recall  f1-score   support

           0     0.9123    0.9733    0.9418     50793
           1     0.9001    0.7199    0.8000     16972

    accuracy                         0.9098     67765
   macro avg     0.9062    0.8466    0.8709     67765
weighted avg     0.9092    0.9098    0.9063     67765

Model saved to: models/25pct_tcn\tcn_calving_25pct.pth

Epoch 2 - calving Accuracy: 0.9106
              precision    recall  f1-score   support

           0     0.9041    0.9852    0.9429     50793
           1     0.9393    0.6874    0.7939     16972

    accuracy                         0.9106     67765
   macro avg     0.9217    0.8363    0.8684     67765
weighted avg     0.9129    0.9106    0.9056     67765

Model saved to: models/25pct_tcn\tcn_calving_25pct.pth

Epoch 3 - calving Accuracy: 0.9197
              precision    recall  f1-score   support

           0     0.9161    0.9830    0.9483     50793
           1     0.

C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



Epoch 1 - lameness Accuracy: 0.8514
              precision    recall  f1-score   support

           0     0.8606    0.9568    0.9061     50824
           1     0.8053    0.5353    0.6431     16956

    accuracy                         0.8514     67780
   macro avg     0.8329    0.7460    0.7746     67780
weighted avg     0.8467    0.8514    0.8403     67780

Model saved to: models/25pct_tcn\tcn_lameness_25pct.pth

Epoch 2 - lameness Accuracy: 0.8636
              precision    recall  f1-score   support

           0     0.8503    0.9928    0.9161     50824
           1     0.9565    0.4763    0.6359     16956

    accuracy                         0.8636     67780
   macro avg     0.9034    0.7345    0.7760     67780
weighted avg     0.8769    0.8636    0.8460     67780

Model saved to: models/25pct_tcn\tcn_lameness_25pct.pth

Epoch 3 - lameness Accuracy: 0.8656
              precision    recall  f1-score   support

           0     0.8563    0.9861    0.9167     50824
           1  

C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



Epoch 1 - mastitis Accuracy: 0.8542
              precision    recall  f1-score   support

           0     0.8465    0.9841    0.9101     50882
           1     0.9069    0.4643    0.6142     16952

    accuracy                         0.8542     67834
   macro avg     0.8767    0.7242    0.7622     67834
weighted avg     0.8616    0.8542    0.8362     67834

Model saved to: models/25pct_tcn\tcn_mastitis_25pct.pth

Epoch 2 - mastitis Accuracy: 0.8641
              precision    recall  f1-score   support

           0     0.8548    0.9863    0.9158     50882
           1     0.9234    0.4972    0.6464     16952

    accuracy                         0.8641     67834
   macro avg     0.8891    0.7417    0.7811     67834
weighted avg     0.8720    0.8641    0.8485     67834

Model saved to: models/25pct_tcn\tcn_mastitis_25pct.pth

Epoch 3 - mastitis Accuracy: 0.8700
              precision    recall  f1-score   support

           0     0.8701    0.9717    0.9181     50882
           1  

#### HYBRID ENSEMBLE MODEL

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from pytorch_tabnet.tab_model import TabNetClassifier
import joblib
import pickle

# ----------------------
# Configuration
# ----------------------
SEQ_LEN = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGETS = ["oestrus", "calving", "lameness", "mastitis"]
FEATURES = ['IN_ALLEYS','REST','EAT','ACTIVITY_LEVEL','hour_sin','hour_cos','eat_rest_ratio','activity_rest_ratio']

model_dirs = {
    "rf": "models/25pct",
    "lgbm": "models/25pct",
    "tabnet": "models/25pct_tabnet",
    "lstm": "models/25pct_lstm",
    "tcn": "models/25pct_tcn",
    "mlp": "models/25pct_mlp",
    "ensemble": "models/25pct_ensemble"
}

# ----------------------
# Helper Classes
# ----------------------
class LSTMModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=64, batch_first=True)
        self.fc = nn.Linear(64, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

class MLPModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        return self.model(x)

class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout):
        super().__init__()
        self.conv1 = nn.utils.weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size,
                                                    stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.utils.weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size,
                                                    stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, input_size=1, output_size=2, num_channels=[16, 32, 64], kernel_size=3, dropout=0.3):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = input_size if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers.append(TemporalBlock(in_channels, out_channels, kernel_size, stride=1,
                                        dilation=dilation_size, padding=(kernel_size-1)*dilation_size,
                                        dropout=dropout))
        self.network = nn.Sequential(*layers)
        self.linear = nn.Linear(num_channels[-1], output_size)

    def forward(self, x):
        x = self.network(x)
        x = x[:, :, -1]
        return self.linear(x)

# ----------------------
# Inference Loop
# ----------------------
for target in TARGETS:
    print(f"\n--- Hybrid Ensemble (25% ADASYN) for: {target} ---")

    # Load Data
    df = pd.read_csv(f"adasyn_25pct/adasyn_25pct_{target}.csv")
    X = df[FEATURES]
    y = df[target]

    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

    # Random Forest
    rf = joblib.load(os.path.join(model_dirs['rf'], f"rf_{target}.pkl"))
    rf_probs = rf.predict_proba(X_test)[:, 1]

    # LightGBM
    lgbm = joblib.load(os.path.join(model_dirs['lgbm'], f"lgbm_{target}.pkl"))
    lgbm_probs = lgbm.predict_proba(X_test)[:, 1]

    # TabNet
    tabnet = TabNetClassifier()
    tabnet.load_model(os.path.join(model_dirs['tabnet'], f"tabnet_{target}_25pct.zip"))
    tabnet_probs = tabnet.predict_proba(X_test.values)[:, 1]

    # MLP
    mlp = MLPModel(input_dim=X.shape[1]).to(DEVICE)
    mlp.load_state_dict(torch.load(os.path.join(model_dirs['mlp'], f"mlp_{target}_25pct.pth"), map_location=DEVICE))
    mlp.eval()
    with torch.no_grad():
        mlp_probs = mlp(torch.tensor(X_test.values, dtype=torch.float32).to(DEVICE)).softmax(dim=1).cpu().numpy()[:, 1]

    # LSTM
    lstm = LSTMModel(input_size=X.shape[1]).to(DEVICE)
    lstm.load_state_dict(torch.load(os.path.join(model_dirs['lstm'], f"lstm_{target}_25pct.pth"), map_location=DEVICE))
    lstm.eval()
    X_seq = X_test.values.reshape(-1, 1, X.shape[1])
    X_seq_tensor = torch.tensor(X_seq, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        lstm_logits = lstm(X_seq_tensor)
        lstm_probs = torch.sigmoid(lstm_logits).cpu().numpy().squeeze()

    # TCN
    tcn = TCNModel()
    tcn.load_state_dict(torch.load(os.path.join(model_dirs['tcn'], f"tcn_{target}_25pct.pth"), map_location=DEVICE))
    tcn.to(DEVICE).eval()
    with torch.no_grad():
        tcn_probs = tcn(X_seq_tensor).softmax(dim=1).cpu().numpy()[:, 1]

    # Combine all predictions
    min_len = min(len(rf_probs), len(lgbm_probs), len(tabnet_probs), len(mlp_probs), len(lstm_probs), len(tcn_probs))
    final_prob = (
        rf_probs[:min_len] +
        lgbm_probs[:min_len] +
        tabnet_probs[:min_len] +
        mlp_probs[:min_len] +
        lstm_probs[:min_len] +
        tcn_probs[:min_len]
    ) / 6

    final_preds = (final_prob > 0.5).astype(int)
    y_true = y_test[:min_len]

    print(classification_report(y_true, final_preds, digits=4))

    # Save ensemble output
    os.makedirs(model_dirs['ensemble'], exist_ok=True)
    output_path = os.path.join(model_dirs['ensemble'], f"ensemble_preds_{target}.pkl")
    with open(output_path, "wb") as f:
        pickle.dump({
            "y_true": y_true,
            "y_pred": final_preds,
            "y_prob": final_prob
        }, f)
    print(f"Saved: {output_path}")



--- Hybrid Ensemble (25% ADASYN) for: oestrus ---


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


              precision    recall  f1-score   support

           0     0.8719    0.9867    0.9258     50699
           1     0.9341    0.5647    0.7039     16887

    accuracy                         0.8813     67586
   macro avg     0.9030    0.7757    0.8148     67586
weighted avg     0.8874    0.8813    0.8703     67586

Saved: models/25pct_ensemble\ensemble_preds_oestrus.pkl

--- Hybrid Ensemble (25% ADASYN) for: calving ---


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


              precision    recall  f1-score   support

           0     0.9558    0.7631    0.8486     50793
           1     0.5578    0.8945    0.6871     16972

    accuracy                         0.7960     67765
   macro avg     0.7568    0.8288    0.7679     67765
weighted avg     0.8561    0.7960    0.8082     67765

Saved: models/25pct_ensemble\ensemble_preds_calving.pkl

--- Hybrid Ensemble (25% ADASYN) for: lameness ---


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


              precision    recall  f1-score   support

           0     0.8189    0.9958    0.8987     50824
           1     0.9639    0.3398    0.5024     16956

    accuracy                         0.8316     67780
   macro avg     0.8914    0.6678    0.7006     67780
weighted avg     0.8551    0.8316    0.7996     67780

Saved: models/25pct_ensemble\ensemble_preds_lameness.pkl

--- Hybrid Ensemble (25% ADASYN) for: mastitis ---


C:\Users\vishn\anaconda3\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
C:\Users\vishn\anaconda3\Lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


              precision    recall  f1-score   support

           0     0.9876    0.9220    0.9537     50882
           1     0.8048    0.9652    0.8777     16952

    accuracy                         0.9328     67834
   macro avg     0.8962    0.9436    0.9157     67834
weighted avg     0.9419    0.9328    0.9347     67834

Saved: models/25pct_ensemble\ensemble_preds_mastitis.pkl
